# Public repository note

This notebook is an output-cleared code copy. It requires locally authorised data and is not runnable from the public repository alone. Green Street raw data, intermediate files, derived aggregates and outputs are not distributed.


# 01 Data Audit and Spatial Framework

Purpose: create the dissertation's data audit and spatial matching foundation without producing final models yet.

This notebook keeps all intermediate results in memory. It does **not** overwrite or export project files.

In [ ]:
from pathlib import Path
import os
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely import wkt
from shapely.geometry import Point

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)

BASE = Path(os.environ.get("DISSERTATION_WORKSPACE", Path.cwd().resolve()))
print(BASE)

## 1. Data Inventory

This section records the role, time period, spatial unit and key fields of each dataset used in the project. The table is a working audit for Chapter 3 Data and Chapter 4 Methodology.

In [ ]:
files = {
    'tfl_2019': 'AnnualisedEntryExit_2019.csv',
    'tfl_2023': 'AC2023_AnnualisedEntryExit.csv',
    'tfl_2024': 'AC2024_AnnualisedEntryExit_Public.csv',
    'tfl_2025': 'AC2025_AnnualisedEntryExit_public.csv',
    'tfl_top100': 'Top100_Master_Spatial_OD_Sheet_Clean.csv',
    'ons_od': 'ODWP01EW_LTLA.csv',
    'lad_boundaries': 'Local_Authority_Districts_May_2024_Boundaries_UK_BGC_3503156029110784919.geojson',
    'underground_stations': 'Underground_Stations.geojson',
    'office_markets': 'London_Office_Markets_V1.geojson',
    'greenstreet_vacancy_2019_2024': 'Greater_London_2019 to 2024 Vacancy - 2026-07-01-1218.csv',
    'greenstreet_vacancy_2026': 'Greater_London_2026_Vacancy_2026-07-01-1222.csv',
    'greenstreet_poi_current': 'Greater_London_POI_Churn_2026-07-01-1110.csv',
    'greenstreet_closed_12m': 'Greater_London_POI_Live_2026-07-01-1110(in).csv',
    'greenstreet_dictionary': 'UK-Retail-Analytics-Pro-Data-Dictionary.pdf',
    'openlocal_parquet': '2025-12-31-shuting-yang-greenstreet-retail.parquet',
}

inventory_rows = []
for key, fname in files.items():
    p = BASE / fname
    inventory_rows.append({
        'dataset_key': key,
        'file': fname,
        'exists': p.exists(),
        'size_mb': round(p.stat().st_size / 1024 / 1024, 2) if p.exists() else np.nan,
    })
file_inventory = pd.DataFrame(inventory_rows)
file_inventory

In [ ]:
data_inventory = pd.DataFrame([
    {'dataset':'TfL station entries/exits', 'file':'AnnualisedEntryExit_2019.csv; AC2023-2025 files', 'years':'2019, 2023, 2024, 2025', 'spatial_unit':'Station', 'key_fields':'Station, Monday, Midweek, Friday entries/exits', 'role':'Construct commuter-shock baseline and post-pandemic Monday/Friday collapse.'},
    {'dataset':'Top 100 commuter-collapse stations', 'file':'Top100_Master_Spatial_OD_Sheet_Clean.csv', 'years':'Derived from 2019, 2023-2025', 'spatial_unit':'Station point + LAD', 'key_fields':'clean_name, cumulative_collapse_score, LAD24NM, top origin LADs', 'role':'Destination station screening and workplace-origin LAD linkage.'},
    {'dataset':'ONS OD commuting matrix', 'file':'ODWP01EW_LTLA.csv', 'years':'Census-based', 'spatial_unit':'LAD-to-LAD', 'key_fields':'origin LAD, workplace LAD, Count', 'role':'Identify residential origin LADs for affected workplace LADs.'},
    {'dataset':'London office market polygons', 'file':'London_Office_Markets_V1.geojson', 'years':'Static boundary', 'spatial_unit':'Office submarket polygon', 'key_fields':'Market, geometry', 'role':'Define five conceptual office submarket groups.'},
    {'dataset':'Green Street vacancy panel', 'file':'Greater_London_2019 to 2024 Vacancy...', 'years':'2019-2024 annual June snapshots', 'spatial_unit':'High street property point', 'key_fields':'RATE_VACANT, RATE_VACANT_LT, NUMBER_UNIT, SCORE_HEALTH_INDEX', 'role':'Retail health/vacancy outcome trends.'},
    {'dataset':'Green Street 2026 vacancy snapshot', 'file':'Greater_London_2026_Vacancy...', 'years':'2026 current snapshot', 'spatial_unit':'High street property point', 'key_fields':'RATE_VACANT, RATE_VACANT_LT, SCORE_HEALTH_INDEX, SCORE_TAP_SCORE', 'role':'Latest retail condition and TAP/Health indicators.'},
    {'dataset':'Green Street current POI/churn file', 'file':'Greater_London_POI_Churn_2026-07-01-1110.csv', 'years':'2026 current snapshot', 'spatial_unit':'POI / premise point', 'key_fields':'TENANT_STATUS, PREMISES_STATUS, CATEGORY, DATE_CREATE, FLAG_IS_NEW', 'role':'Current live/vacant/demolished retail structure.'},
    {'dataset':'Green Street closed records', 'file':'Greater_London_POI_Live_2026-07-01-1110(in).csv', 'years':'Closures over last 12 months', 'spatial_unit':'POI / premise point', 'key_fields':'TENANT_STATUS=Closed, DATE_CLOSE, CATEGORY, SUBCATEGORY', 'role':'Recent closure/churn by geography and category.'},
    {'dataset':'OpenLocal property data', 'file':'2025-12-31-shuting-yang-greenstreet-retail.parquet', 'years':'2019Q1-2025Q4', 'spatial_unit':'Property record by LAD/period', 'key_fields':'category_group, occupation_state, floor area, rateable value, account name', 'role':'Office restructuring, retail adaptation, and validation indicators.'},
])
data_inventory

## 2. Build the Five Office Submarket Framework

The ArcGIS layer contains 26 detailed office market polygons. These are grouped into five dissertation submarkets aligned with the Green Street brief and Avison Young-style Central London office market definitions.

In [ ]:
office = gpd.read_file(BASE / files['office_markets']).to_crs('EPSG:4326')
print('Office market polygons:', len(office))
print('CRS:', office.crs)
sorted(office['Market'].dropna().unique())

In [ ]:
# Working crosswalk. Keep outside-core markets in the data instead of deleting them.
submarket_crosswalk = {
    # West End
    'Mayfair': 'West End',
    'Soho': 'West End',
    "St James's": 'West End',
    'Covent Garden': 'West End',
    'Fitzrovia': 'West End',
    'North of Oxford Street': 'West End',
    'Paddington': 'West End',
    'Knightsbridge': 'West End',
    'Victoria': 'West End',

    # City
    'City Core': 'City',

    # Tech Belt & Midtown
    'Midtown': 'Tech Belt & Midtown',
    'Bloomsbury': 'Tech Belt & Midtown',
    'Clerkenwell': 'Tech Belt & Midtown',
    'Euston': 'Tech Belt & Midtown',
    'Kings Cross': 'Tech Belt & Midtown',
    'Shoreditch': 'Tech Belt & Midtown',
    'Camden': 'Tech Belt & Midtown',
    'Aldgate & Whitechapel': 'Tech Belt & Midtown',

    # Canary Wharf
    'Canary Wharf': 'Canary Wharf',

    # Southbank
    'Southbank': 'Southbank',
    'Waterloo': 'Southbank',
    'Vauxhall, Nine Elms and Battersea': 'Southbank',
}

office['study_submarket'] = office['Market'].map(submarket_crosswalk).fillna('Outside core / comparison')
office['inside_core_submarket'] = office['study_submarket'].ne('Outside core / comparison')

crosswalk_table = office[['Market','study_submarket','inside_core_submarket']].sort_values(['inside_core_submarket','study_submarket','Market'], ascending=[False, True, True])
crosswalk_table

In [ ]:
office_5 = office[office['inside_core_submarket']].dissolve(by='study_submarket', as_index=False)
print('Five core submarket geometries:', len(office_5))
office_5[['study_submarket','geometry']]

In [ ]:
ax = office.plot(column='study_submarket', figsize=(10, 7), edgecolor='white', linewidth=0.8, legend=True, alpha=0.85)
ax.set_title('Working Office Submarket Framework')
ax.set_axis_off()

## 3. Match TfL Commuter-Collapse Stations to LADs and Office Submarkets

The station layer measures the commuter-demand shock. Stations are assigned to both office submarkets and LADs so that the same result can support Green Street submarket analysis and OpenLocal LAD-based analysis.

In [ ]:
top100 = pd.read_csv(BASE / files['tfl_top100'])

# Robust WKT parsing: keep rows with valid station geometry and record any missing cases.
def parse_wkt_or_none(value):
    if isinstance(value, str) and value.strip():
        try:
            return wkt.loads(value)
        except Exception:
            return None
    return None

top100['geometry_parsed'] = top100['geometry'].apply(parse_wkt_or_none)
top100_missing_geometry = top100[top100['geometry_parsed'].isna()][['clean_name','geometry']].copy()
print('Rows with missing/invalid geometry:', len(top100_missing_geometry))

top100_valid = top100.dropna(subset=['geometry_parsed']).copy()
top100_gdf = gpd.GeoDataFrame(top100_valid.drop(columns=['geometry']), geometry='geometry_parsed', crs='EPSG:4326')
top100_gdf = top100_gdf.rename_geometry('geometry')

# Strict point-in-polygon assignment.
top100_sub = gpd.sjoin(
    top100_gdf,
    office[['Market','study_submarket','inside_core_submarket','geometry']],
    how='left',
    predicate='intersects'
).drop(columns=['index_right'])

top100_sub['study_submarket'] = top100_sub['study_submarket'].fillna('Outside office market polygon')
top100_sub['inside_core_submarket'] = top100_sub['inside_core_submarket'].fillna(False)

print('Top 100 rows:', len(top100))
print('Rows used for spatial matching:', len(top100_sub))
print('Stations inside five core submarkets:', int(top100_sub['inside_core_submarket'].sum()))
top100_sub[['clean_name','cumulative_collapse_score','LAD24NM','Market','study_submarket','inside_core_submarket']].head(15)

In [ ]:
# 800m buffered assignment: useful for stations just outside a market polygon but serving it.
stations_27700 = top100_gdf.to_crs('EPSG:27700').copy()
stations_27700['geometry'] = stations_27700.buffer(800)
office_27700 = office[['Market','study_submarket','inside_core_submarket','geometry']].to_crs('EPSG:27700')

buffer_join = gpd.sjoin(stations_27700, office_27700, how='left', predicate='intersects')
buffer_summary = (
    buffer_join.dropna(subset=['study_submarket'])
    .groupby('clean_name')
    .agg(buffer_markets=('Market', lambda x: '; '.join(sorted(set(x)))),
         buffer_submarkets=('study_submarket', lambda x: '; '.join(sorted(set(x)))),
         buffer_core=('inside_core_submarket', 'max'))
    .reset_index()
)

top100_matched = top100_sub.merge(buffer_summary, on='clean_name', how='left')
top100_matched['buffer_core'] = top100_matched['buffer_core'].fillna(False)

top100_matched[['clean_name','cumulative_collapse_score','LAD24NM','study_submarket','inside_core_submarket','buffer_submarkets','buffer_core']].head(20)

In [ ]:
tfl_submarket_summary = (
    top100_matched
    .groupby('study_submarket', dropna=False)
    .agg(stations=('clean_name','count'),
         mean_collapse=('cumulative_collapse_score','mean'),
         max_collapse=('cumulative_collapse_score','max'))
    .sort_values('mean_collapse', ascending=False)
)
tfl_lad_summary = (
    top100_matched
    .groupby('LAD24NM', dropna=False)
    .agg(stations=('clean_name','count'),
         mean_collapse=('cumulative_collapse_score','mean'),
         max_collapse=('cumulative_collapse_score','max'))
    .sort_values(['stations','mean_collapse'], ascending=False)
)

display(tfl_submarket_summary)
display(tfl_lad_summary.head(20))

In [ ]:
origin_cols = ['Origin_LAD_1_Name','Origin_LAD_2_Name','Origin_LAD_3_Name']
origin_counts = pd.concat([top100[c] for c in origin_cols]).dropna().value_counts().rename_axis('origin_lad').reset_index(name='frequency')
workplace_lads = top100[['LAD24CD','LAD24NM']].drop_duplicates().rename(columns={'LAD24CD':'lad_code','LAD24NM':'lad_name'})
origin_counts.head(20), workplace_lads.sort_values('lad_name').head(20)

## 4. Match Green Street Vacancy and POI Data to Submarkets and LADs

Green Street records have coordinates, so they can be joined directly to office submarket polygons. They are also joined to LADs to support validation against OpenLocal data.

In [ ]:
lad = gpd.read_file(BASE / files['lad_boundaries']).to_crs('EPSG:4326')
print('LAD polygons:', len(lad), 'CRS:', lad.crs)
print([c for c in lad.columns if c.startswith('LAD24')])

In [ ]:
def points_from_lonlat(df, lon='LONGITUDE', lat='LATITUDE', crs='EPSG:4326'):
    clean = df.dropna(subset=[lon, lat]).copy()
    return gpd.GeoDataFrame(clean, geometry=gpd.points_from_xy(clean[lon], clean[lat]), crs=crs)

def assign_geographies(points_gdf, label):
    sub = gpd.sjoin(points_gdf, office[['Market','study_submarket','inside_core_submarket','geometry']], how='left', predicate='intersects').drop(columns=['index_right'])
    sub['study_submarket'] = sub['study_submarket'].fillna('Outside office market polygon')
    sub['inside_core_submarket'] = sub['inside_core_submarket'].fillna(False)
    sub = gpd.sjoin(sub, lad[['LAD24CD','LAD24NM','geometry']], how='left', predicate='intersects').drop(columns=['index_right'])
    print(label, 'rows:', len(sub), 'inside core:', int(sub['inside_core_submarket'].sum()), 'with LAD:', sub['LAD24NM'].notna().sum())
    return sub

In [ ]:
gs_hist = pd.read_csv(BASE / files['greenstreet_vacancy_2019_2024'], low_memory=False)
gs_hist['year'] = pd.to_datetime(gs_hist['DATE_EOM']).dt.year
gs_hist_gdf = points_from_lonlat(gs_hist)
gs_hist_match = assign_geographies(gs_hist_gdf, 'Green Street historical vacancy')

hist_summary = (
    gs_hist_match
    .groupby(['year','study_submarket'])
    .agg(properties=('PROPERTY_ID','nunique'),
         mean_vacancy=('RATE_VACANT','mean'),
         mean_long_term_vacancy=('RATE_VACANT_LT','mean'),
         mean_health_index=('SCORE_HEALTH_INDEX','mean'),
         total_units=('NUMBER_UNIT','sum'),
         total_vacant_units=('NUMBER_UNIT_VACANT','sum'))
    .reset_index()
    .sort_values(['study_submarket','year'])
)
hist_summary.head(20)

In [ ]:
gs_2026 = pd.read_csv(BASE / files['greenstreet_vacancy_2026'], low_memory=False)
gs_2026['year'] = 2026
gs_2026_gdf = points_from_lonlat(gs_2026)
gs_2026_match = assign_geographies(gs_2026_gdf, 'Green Street 2026 vacancy')

current_summary = (
    gs_2026_match
    .groupby('study_submarket')
    .agg(properties=('PROPERTY_ID','nunique'),
         mean_vacancy=('RATE_VACANT','mean'),
         mean_long_term_vacancy=('RATE_VACANT_LT','mean'),
         mean_health_index=('SCORE_HEALTH_INDEX','mean'),
         mean_tap=('SCORE_TAP_SCORE','mean'),
         total_units=('NUMBER_UNIT','sum'),
         total_vacant_units=('NUMBER_UNIT_VACANT','sum'))
    .sort_values('mean_vacancy', ascending=False)
)
current_summary

In [ ]:
poi_cols = ['AUTO_ID','TENANT_ID','PREMISES_ID','PROPERTY_ID','TENANT_STATUS','PREMISES_STATUS','CLASSIFICATION','CATEGORY','SUBCATEGORY','GEOGRAPHY','PROPERTY','LATITUDE','LONGITUDE','AREA_SM','VOA_BUSINESS_RATE','ADDRESS','ZIP','DATE_PREMISES_CREATE','DATE_CREATE','DATE_CLOSE','DATE_LAST_SURVEY_OFFICE','DATE_LAST_SURVEY_FIELD','RATE_PROPERTY_VACANT','RATE_PROPERTY_VACANT_LT','SCORE_HEALTH_INDEX','SCORE_TAP_SCORE','FLAG_IS_NEW']
gs_poi = pd.read_csv(BASE / files['greenstreet_poi_current'], usecols=lambda c: c in poi_cols, low_memory=False)
gs_poi_gdf = points_from_lonlat(gs_poi)
gs_poi_match = assign_geographies(gs_poi_gdf, 'Green Street current POI')

poi_status_summary = pd.crosstab(gs_poi_match['study_submarket'], gs_poi_match['TENANT_STATUS'], margins=True).sort_index()
poi_status_summary

In [ ]:
closed_cols = ['AUTO_ID','TENANT_ID','PREMISES_ID','PROPERTY_ID','TENANT','TENANT_STATUS','PREMISES_STATUS','CLASSIFICATION','CATEGORY','SUBCATEGORY','GEOGRAPHY','PROPERTY','LATITUDE','LONGITUDE','ADDRESS','ZIP','DATE_CREATE','DATE_CLOSE','DATE_LAST_SURVEY_OFFICE','DATE_LAST_SURVEY_FIELD','RATE_PROPERTY_VACANT','RATE_PROPERTY_VACANT_LT','SCORE_HEALTH_INDEX','SCORE_TAP_SCORE','FLAG_IS_NEW']
gs_closed = pd.read_csv(BASE / files['greenstreet_closed_12m'], encoding='cp1252', usecols=lambda c: c in closed_cols, low_memory=False)
gs_closed['close_month'] = pd.to_datetime(gs_closed['DATE_CLOSE'], errors='coerce').dt.to_period('M').astype(str)
gs_closed_gdf = points_from_lonlat(gs_closed)
gs_closed_match = assign_geographies(gs_closed_gdf, 'Green Street closed records')

closed_summary = (
    gs_closed_match
    .groupby(['study_submarket','CLASSIFICATION'])
    .agg(closures=('TENANT_ID','count'),
         unique_premises=('PREMISES_ID','nunique'))
    .reset_index()
    .sort_values(['study_submarket','closures'], ascending=[True, False])
)
closed_summary.head(30)

## 5. OpenLocal Parquet: Basic Retail and Office Indicators

This cell constructs lightweight aggregate indicators by LAD and period. These are not final models; they show whether H1/H2/H3 can be operationalised from the available fields.

In [ ]:
openlocal_cols = ['period','geocode_name','geocode','category_group','uarn','account_name','occupation_state','total_floor_area','rateable_value','unadjusted_price']
openlocal = pd.read_parquet(BASE / files['openlocal_parquet'], columns=openlocal_cols)
openlocal['period'] = pd.to_datetime(openlocal['period'])
openlocal['year'] = openlocal['period'].dt.year

openlocal_core = openlocal[openlocal['category_group'].isin(['RETAIL','OFFICE'])].copy()
openlocal_agg = (
    openlocal_core
    .groupby(['period','year','geocode_name','geocode','category_group'])
    .agg(records=('uarn','size'),
         unique_units=('uarn','nunique'),
         accounts=('account_name','nunique'),
         occupied_records=('occupation_state', lambda s: (s == 'OCCUPIED').sum()),
         vacant_records=('occupation_state', lambda s: (s == 'VACANT').sum()),
         total_floor_area=('total_floor_area','sum'),
         median_floor_area=('total_floor_area','median'),
         total_rateable_value=('rateable_value','sum'),
         median_price=('unadjusted_price','median'))
    .reset_index()
)
openlocal_agg['vacancy_share_records'] = openlocal_agg['vacant_records'] / (openlocal_agg['occupied_records'] + openlocal_agg['vacant_records'])

print('OpenLocal aggregate rows:', len(openlocal_agg))
openlocal_agg.head(10)

In [ ]:
# Office restructuring indicators by LAD/year, currently using annual averages over quarters.
office_year = (
    openlocal_agg[openlocal_agg['category_group'].eq('OFFICE')]
    .groupby(['year','geocode_name','geocode'])
    .agg(office_units=('unique_units','mean'),
         office_total_floor=('total_floor_area','mean'),
         office_median_floor=('median_floor_area','mean'),
         office_vacancy=('vacancy_share_records','mean'))
    .reset_index()
)
retail_year = (
    openlocal_agg[openlocal_agg['category_group'].eq('RETAIL')]
    .groupby(['year','geocode_name','geocode'])
    .agg(retail_units=('unique_units','mean'),
         retail_total_floor=('total_floor_area','mean'),
         retail_median_floor=('median_floor_area','mean'),
         retail_vacancy=('vacancy_share_records','mean'),
         retail_rateable_value=('total_rateable_value','mean'))
    .reset_index()
)

openlocal_year = office_year.merge(retail_year, on=['year','geocode_name','geocode'], how='outer')
openlocal_year.head(10)

## 6. Feasibility Check for Research Expectations

This table links the current data inventory to the dissertation's three analytical expectations. It should be revised after the first exploratory plots and any 2025 Green Street update.

In [ ]:
feasibility = pd.DataFrame([
    {
        'expectation':'H1: commuter decline and retail adaptation',
        'needed_variables':'TfL commuter shock; workplace retail vitality; residential-origin retail indicators',
        'available_data':'Top100 station collapse; ONS OD workplace-origin LADs; Green Street vacancy/Health; OpenLocal retail floor area, vacancy, rateable value',
        'main_spatial_unit':'Station -> submarket for workplace; LAD for OD/OpenLocal origin areas',
        'years':'TfL 2019/2023-2025; Green Street 2019-2024 + 2026; OpenLocal 2019-2025',
        'current_feasibility':'Feasible, but origin-area adaptation should use multiple indicators rather than only openings/closures.',
    },
    {
        'expectation':'H2: office restructuring and retail resilience',
        'needed_variables':'Office floor area distribution; office total floor area; retail vacancy/health outcomes',
        'available_data':'OpenLocal OFFICE floor area and vacancy by LAD/period; Green Street vacancy and Health Index by property/submarket',
        'main_spatial_unit':'LAD for OpenLocal; submarket/property for Green Street; bridge via station/submarket/LAD labels',
        'years':'OpenLocal 2019-2025; Green Street 2019-2024 + 2026',
        'current_feasibility':'Strong candidate for core analysis; need to define restructuring metrics carefully.',
    },
    {
        'expectation':'H3: volatility vs stagnation',
        'needed_variables':'Closures, openings/new POIs, churn, long-term vacancy, volume change',
        'available_data':'Green Street closed records over last 12 months; current POI live/vacant/demolished; long-term vacancy; OpenLocal account and floor area changes',
        'main_spatial_unit':'Submarket/property for Green Street; LAD for OpenLocal robustness',
        'years':'Green Street recent/current; OpenLocal 2019-2025',
        'current_feasibility':'Feasible as pathway/typology analysis, not as a claim that churn always increases.',
    },
    {
        'expectation':'Green Street vs OpenLocal validation',
        'needed_variables':'Comparable vacancy/retail health measures across same geographies',
        'available_data':'Green Street vacancy/long-term vacancy/Health Index; OpenLocal occupation_state-derived vacancy and retail indicators',
        'main_spatial_unit':'LAD validation pool from affected workplace LADs + origin LADs; submarket labels as interpretation layer',
        'years':'Best common periods currently 2019, 2023, 2024; 2025 Green Street would improve alignment',
        'current_feasibility':'Feasible; run first as correlation/trend agreement rather than proof of equivalence.',
    },
])
feasibility

## 7. Immediate Next Checks

1. Confirm the final crosswalk for the five office submarkets.
2. Inspect stations outside strict polygons but inside 800m buffers.
3. Plot Green Street vacancy trend by submarket.
4. Build LAD-level validation pool from affected workplace LADs and origin LADs.
5. Decide whether 2025 Green Street vacancy is necessary after seeing common-year results.